<a href="https://colab.research.google.com/github/LuciaMellini/AMD_project/blob/main/findingSimilarItems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding similar items

We download the Letterboxd dataset from Kaggle, using a token.

In [279]:
import os
import json
import pandas as pd
import pip
import string
import re
import numpy as np

os.environ['KAGGLE_USERNAME'] = "xxx"
os.environ['KAGGLE_KEY'] = "xxx"

In [280]:
#! kaggle datasets download -d gsimonx37/letterboxd

We only consider a subset of the files contained in the `letterboxd` dataset, namely the data regarding the movie names and ids, their actors, crews, genres and themes.

In [281]:
# import zipfile
# from multiprocessing import Pool

DATA_DIR = "./letterboxd"
# members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
# with zipfile.ZipFile(DATA_DIR + ".zip","r") as zip_ref:
#     for file_name in members_to_extract:
#         zip_ref.extract(file_name + '.csv', DATA_DIR)
!tar xf ./drive/MyDrive/letterboxd.tar.gz

We then prepare the entry point for the Spark functionalities that will we use from now on.

In [282]:
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
#!wget https://downloads.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
!tar xf ./drive/MyDrive/spark-3.5.3-bin-hadoop3.tgz
!pip install -q findspark

In [283]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = "./drive/MyDrive/spark-3.5.3-bin-hadoop3"

import findspark
findspark.init("spark-3.5.3-bin-hadoop3")
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.executor.memory", "2g") \
    .appName("ColabSpark") \
    .getOrCreate()

sc = spark.sparkContext

We begin by getting the input files into RDD form.

In [284]:
SUB_FUNC = lambda s: re.sub(r'(\d+),[\s\t]+([a-zA-Z])', r'\1,\2', s)      # some actor names are written with a heading space, we exploit that the preceeding attribute is numerical (the movie id)
STRIP_FUNC = lambda s: s.strip('"')

def csv_to_rdd(filename):
    raw = sc.textFile(filename)
    result = (raw
              .map(STRIP_FUNC)
              .map(SUB_FUNC)
              .map(lambda r: re.split(r',(?! )', r)))  #split only on commas that are followed by a character to avoid splitting sentences e.g. in movie descriptions
    return result

def get_column_names(rdd):
    column_names = rdd.filter(lambda r: r[0]=='id').collect()[0][1:]
    return column_names

def prepare_data(filename):
    rdd = csv_to_rdd(filename).cache()

    column_names = get_column_names(rdd)
    result = (rdd
            .filter(lambda r: r[0]!='id')
            .map(lambda r: (int(r[0]), dict(zip(column_names, r[1:])))))
    return result


In [285]:
members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
letterboxd_RDDs={}

for member in members_to_extract:
    letterboxd_RDDs[member] = prepare_data(os.path.join(DATA_DIR, f"{member}.csv"))
    print(letterboxd_RDDs[member].first())

(1000001, {'name': 'Margot Robbie', 'role': 'Barbie'})
(1000001, {'role': 'Director', 'name': 'Greta Gerwig'})
(1000001, {'genre': 'Comedy'})
(1000001, {'name': 'Barbie', 'date': '2023', 'tagline': "She's everything. He's just Ken.", 'description': '"Barbie and Ken are having the time of their lives in the colorful and seemingly perfect world of Barbie Land. However, when they get a chance to go to the real world, they soon discover the joys and perils of living among humans."', 'minute': '114', 'rating': '3.86'})
(1000001, {'theme': 'Humanity and the world around us'})


In [286]:
def get_sample(rdd, size):
    """ Extract a sample of records from the RDD based on a specified size

    Args:
        rdd (pyspark.RDD): The input RDD containing records, where each record's first element is expected
                           to be an ID as a string.
        size (int): The desired number of records to sample. The function filters records with IDs less
                    than or equal to 1,000,000 plus the specified size.

    Returns:
        pyspark.RDD: An RDD containing the filtered sample of records.
    """
    return rdd.filter(lambda r: r[0]<=1000000+size)

Below we have prepared a function to extract a sample of the data, based on the ids in the datasets. The maximum size of the sample is $125641$.




In [287]:
%%time
sample_size = 100
sample_size_bc = sc.broadcast(sample_size)
letterboxd_RDDs_sample = {}
for member in members_to_extract:
    letterboxd_RDDs[member] = get_sample(letterboxd_RDDs[member], sample_size_bc.value)

CPU times: user 1.98 ms, sys: 908 µs, total: 2.89 ms
Wall time: 23.7 ms


In [288]:
letterboxd_RDDs['actors'].first()

(1000001, {'name': 'Margot Robbie', 'role': 'Barbie'})

Let's look at the amount of rows for each member.

In [289]:
for member in members_to_extract:
    print(f"Number of rows for {member}:\t{letterboxd_RDDs[member].count()}")

Number of rows for actors:	6130
Number of rows for crew:	9115
Number of rows for genres:	275
Number of rows for movies:	100
Number of rows for themes:	704


A glimpse at the structure of the rows in the RDDs of each member.

In [290]:
for member in members_to_extract:
    print(f"Row for {member:}:\t {letterboxd_RDDs[member].first()}")

Row for actors:	 (1000001, {'name': 'Margot Robbie', 'role': 'Barbie'})
Row for crew:	 (1000001, {'role': 'Director', 'name': 'Greta Gerwig'})
Row for genres:	 (1000001, {'genre': 'Comedy'})
Row for movies:	 (1000001, {'name': 'Barbie', 'date': '2023', 'tagline': "She's everything. He's just Ken.", 'description': '"Barbie and Ken are having the time of their lives in the colorful and seemingly perfect world of Barbie Land. However, when they get a chance to go to the real world, they soon discover the joys and perils of living among humans."', 'minute': '114', 'rating': '3.86'})
Row for themes:	 (1000001, {'theme': 'Humanity and the world around us'})


For each member the available attributes are the following:

| **Member**    | **Attributes**                             |
|--------------|-----------------------------------------|
| **actor**    | name, role                          |
| **crew**     | role, name                          |
| **genres**   | genre                               |
| **movies**   | name, date, tagline, description, minute, rating |
| **themes**   | theme                               |
</br>

For this project we would like to focus on the following features:
<a name="table1"></a>

| **Member**    | **Attributes**                             |
|--------------|-----------------------------------------|
| **actor**    | names of the first 10 actors for a given movie                      |
| **crew**     | name(s) of the director of each movie                        |
| **genres**   | genre                               |
| **movies**   | name, date, minute, rating |
| **themes**   | theme                               |


We set up some primitives to manipulate the dictionaries that are the values in the RDDs' rows.

In [291]:
reduce_dicts = lambda x, y: {
        key: (x[key] + [y[key]] if isinstance(x[key], list) else [x[key], y[key]])
        for key in x.keys()
    }

remove_dict_item = lambda d, key: (d.pop(key), d)[1] if key in d else d

rename_key_in_dict = lambda d, old_key, new_key: remove_dict_item({**d, new_key: d.pop(old_key)} if old_key in d else d, old_key)

update_dict_value = lambda d, key, new_value: {**d, key: new_value}

We filter only the directors from the crew dataset, and we ony keep their name.

In [292]:
letterboxd_RDDs['crew'] = (letterboxd_RDDs['crew']
                            .filter(lambda r: r[1]['role']=='Director')
                            .map(lambda r: (r[0], rename_key_in_dict(r[1], 'name', 'director')))
                            .map(lambda r: (r[0], remove_dict_item(r[1], 'role'))))

In [293]:
letterboxd_RDDs['crew'].take(5)

[(1000001, {'director': 'Greta Gerwig'}),
 (1000002, {'director': 'Bong Joon-ho'}),
 (1000003, {'director': 'Daniel Scheinert'}),
 (1000003, {'director': 'Daniel Kwan'}),
 (1000004, {'director': 'David Fincher'})]

For each category we account for movies having multiple values for a given attribute.

In [294]:
for member in members_to_extract:
    letterboxd_RDDs[member] = letterboxd_RDDs[member].reduceByKey(lambda a, b: reduce_dicts(a, b))

We keep only the `n_actors` most relevant actors in each movie.

In [295]:
n_actors = 6
letterboxd_RDDs['actors'] = (letterboxd_RDDs['actors']
                            .map(lambda r: (r[0], update_dict_value(r[1], 'name', r[1]['name'][:n_actors]))))

We only keep the actors' names, and create a field for each of the `n_actors` selected actors.

In [296]:
letterboxd_RDDs['actors'] = (letterboxd_RDDs['actors']
                            .map(lambda r: (r[0], rename_key_in_dict(r[1], 'name', 'actors')))
                            .map(lambda r: (r[0], remove_dict_item(r[1], 'role')))
                            .map(lambda r: (r[0], {**r[1],**{f'actor{i+1}': actor for i, actor in enumerate(r[1]['actors'])}}))
                            .map(lambda r: (r[0], remove_dict_item(r[1], 'actors'))))

For example,

In [297]:
letterboxd_RDDs['actors'].first()

(1000002,
 {'actor1': 'Song Kang-ho',
  'actor2': 'Lee Sun-kyun',
  'actor3': 'Cho Yeo-jeong',
  'actor4': 'Choi Woo-shik',
  'actor5': 'Park So-dam',
  'actor6': 'Lee Jung-eun'})

For each movie we only store the attributes listed in the <a href="#table1">table above</a>.

In [298]:
letterboxd_RDDs['movies'] = (letterboxd_RDDs['movies']
                            .map(lambda r: (r[0], remove_dict_item(remove_dict_item(r[1], 'tagline'), 'description'))))


In [299]:
letterboxd_RDDs['movies'].first()

(1000008, {'name': 'Joker', 'date': '2019', 'minute': '122', 'rating': '3.85'})

In [300]:
for member in members_to_extract:
    print(letterboxd_RDDs[member].first())

(1000002, {'actor1': 'Song Kang-ho', 'actor2': 'Lee Sun-kyun', 'actor3': 'Cho Yeo-jeong', 'actor4': 'Choi Woo-shik', 'actor5': 'Park So-dam', 'actor6': 'Lee Jung-eun'})
(1000005, {'director': 'Damien Chazelle'})
(1000002, {'genre': ['Comedy', 'Thriller', 'Drama']})
(1000008, {'name': 'Joker', 'date': '2019', 'minute': '122', 'rating': '3.85'})
(1000002, {'theme': ['Humanity and the world around us', 'Intense violence and sexual transgression', 'Twisted dark psychological thriller', 'Heartbreaking and moving family drama', 'Enduring stories of family and marital drama', 'Touching and sentimental family stories', 'Intense political and terrorist thrillers']})


In [246]:
movies_RDD = letterboxd_RDDs[members_to_extract[0]]
for member in members_to_extract[1:]:
    movies_RDD = movies_RDD.join(letterboxd_RDDs[member]).mapValues(lambda x: {**x[0], **x[1]})

movies_RDD = (movies_RDD
                .flatMap(lambda x: [((x[0], key), value) for key, value in x[1].items()])
                .map(lambda r: ((int(r[0][0]), r[0][1]), r[1])))

A generic row of `movies_RDD` has the following format:
<p align=center><i>((id, category), list_of_values)</i></p>

For example,

In [247]:
movies_RDD.take(5)

[]

## Data pre-processing

To preserve the independent role of each attribute we have decided to measure their similarity using cosine distance. This entails translating all data regarding a movie into a vector with components in $\mathbb{R}$.

Below we list the data types of the various attributes.

| **Attributes**    | **Datatype**                       |
|--------------|-----------------------------------------|
| **actors**    | string         |
| **director**     | string                        |
| **genre**   | string                             |
| **theme**   | string                             |
| **name**   | string |
| **date**   | numerical |
| **minute**   | numerical |
| **rating**   | numerical                             |

It is evident that the textual attributes have to be transformed into values to be able to work in an Euclidean space. The following paragraphs are dedicated to these transformations. We refer to the report for a discussion regarding the chosen methods.


To simplify the operations that follow we chang the RDD structure such that a row has the following format:
<p align=center><i>((id, category), list_of_values)</i></p>

### String pre-processing

We bring all the strings in data dictionary to lower case, eccept for the names of actors and director. In addition to names always being capitalized, we will not consider them from a semantic point of view, so their uniformation in preparation for the next steps would be useless.

In [ ]:
categories_all = [f'actor{i+1}' for i in range(n_actors)]+['director', 'genre', 'name', 'theme', 'date', 'minute', 'rating']

In [ ]:
movies_hashed_RDD = movies_RDD
movies_RDD = movies_RDD.map(lambda r: (r[0][0], (r[0][1], r[1]))).groupByKey().mapValues(list)

In [ ]:
def apply_to_categories(rdd, l, f):
    """ Apply a function to specific categories within an RDD

    Args:
        rdd (pyspark.RDD): The input RDD containing tuples, where the first element is an identifier and the second element is a list of values.
        l (list): A list of categories (keys) to which the function should be applied.
        f (function): A function to apply to each element of the lists in the second element of the tuples
                      for the specified categories.

    Returns:
        pyspark.RDD: An RDD where the function `f` has been applied to the lists in the second element of
                     the tuples for the specified categories. Other tuples remain unchanged.
    """
    return rdd.map(lambda r: (r[0], [f(s) for s in r[1]]) if r[0][1] in l else r)

In [ ]:
categories_lower = ['genre', 'name', 'theme']
movies_hashed_RDD = apply_to_categories(movies_hashed_RDD, categories_lower, lambda x: x.lower())

To distill the semantics of the movie's theme we apply the following NLP processing steps:
* remove stop words
* replace the words with their lemmatized version

In [ ]:
import spacy
! python -m spacy download en_core_web_md -q

nlp = spacy.load("en_core_web_md")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 21.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from functools import reduce

combine_functions = lambda *funcs: lambda x: reduce(lambda v, f: f(v), funcs, x) #v accumulator value
remove_punctuation = lambda x: re.sub(r'[^\w\s]','',x)
remove_multiple_spaces = lambda x: re.sub(r'\s+',' ',x)
remove_stop_words_func = lambda x: " ".join([token.text for token in nlp(x) if not token.is_stop])
lemmatize_func = lambda x: " ".join([token.lemma_ for token in nlp(x)])

nlp_processing = combine_functions(remove_punctuation, remove_multiple_spaces, remove_stop_words_func, lemmatize_func)

categories_nlp = ['theme']
movies_hashed_RDD = apply_to_categories(movies_hashed_RDD, categories_nlp, nlp_processing)

#### Embedding strings

We use the SpaCy text vectorizations of the attributes:
* name
* genre
* theme

In [ ]:
embedding_func = lambda x: nlp(x).vector

categories_vec = ['name', 'genre', 'theme']
movies_hashed_RDD = apply_to_categories(movies_hashed_RDD, categories_vec, embedding_func)

For example a value for the *name* attribute will result as such:

In [ ]:
id, value = movies_hashed_RDD.filter(lambda r: r[0][1]=='name').first()
print("id:\t {}\nvalue:\t {}".format(id,value[0]))

id:	 (1000026, 'name')
value:	 [-1.3639499   0.4602     -2.1774     -0.79657495  2.4384499   0.87569
  3.3134499   1.8447499  -1.968175   -0.20194499  2.4148002  -0.0869855
 -1.8018498   1.0198901  -0.46349996  1.84985     3.5685      2.7028
  0.07104    -0.26105     0.25559     1.94155    -0.84536004 -0.855579
  0.21219999 -0.69476503  0.16405    -0.57121503  1.4813     -1.2403345
 -2.14388     0.32292497 -0.78045005  0.53714997 -3.95713    -0.22048
  0.03349999 -1.0695     -0.31620002 -0.14294994 -2.47855    -0.617625
  0.84711504  1.94925     1.7835699   0.06661001 -2.9288     -1.27225
  0.89522004  2.86625    -1.9094399   2.55475     0.04369998 -2.826225
 -1.00735     1.264905    1.00991    -0.56691504  1.137905    0.29093498
  2.93195    -0.6550499  -1.62099    -2.2964048   1.33041    -2.13555
 -3.30635    -3.4407501  -0.26583502  0.855115   -0.47981152  0.04334998
 -1.62485    -0.88742995  0.0810695  -0.78427    -3.4417      1.100765
 -0.794195   -0.51276493 -3.02475     1.18116 

#### Hashing strings

We hash the strings of the features:
* actors
* director
person name hash
(no bias towards similar names)

In [ ]:
import math

def hash_object(byte_obj, seed, hash_bucket_size):
    """ Hash an object into a bucket of values [0,hash_bucket_size-1] on the basis of the seed

        Args:
            byte_obj (byte array): an object in byte format
            seed (int): seed for the hash function
            hash_bucket_size (int): the size of the bucket to which the objects get hashed to

        Returns:
            int: hashed object
        """
    m=hashlib.shake_256()
    m.update(byte_obj)
    m.update(bytes(seed))
    required_bytes = math.ceil(hash_bucket_size / 8)
    hash_output = m.digest(required_bytes)
    hashed_value = int.from_bytes(hash_output, 'little')
    return hashed_value % hash_bucket_size

In [ ]:
import hashlib
hash_func = lambda v: hash_object(bytearray(v,'utf-8'),42,2**30)

categories_hash = [f"actor{i+1}" for i in range(n_actors)]+['director']
movies_hashed_RDD = apply_to_categories(movies_hashed_RDD, categories_hash, hash_func)

For example a value for the *director* attribute will result as such:

In [ ]:
id, value = movies_hashed_RDD.filter(lambda r: r[0][1]=='director').first()
print("id:\t {}\nvalue:\t {}".format(id,value[0]))

id:	 (1000001, 'director')
value:	 155972137


### Vector preparation

Now that we have prepared all categories in a targeted way, we can proceed by building the vectors of the movies.

#### Reduce attributes with multiple values

We simply average the values or arrays for attributes that have multiple values.

In [ ]:
vectors_RDD = movies_hashed_RDD.map(lambda r: (r[0], ((np.mean(r[1], axis=0)).tolist() if r[0][1] in categories_vec else [np.mean([float(s) for s in r[1]])])))

In [ ]:
vectors_RDD = vectors_RDD.cache()

#### Standardization

We reduce the amount of features for each vector to avoid suffering from the curse of dimensionality when evaluating their similarity. Before reducing the vectors we standardize their components such that each feature has a mean of $0$ and a standard deviation of $1$.

In [ ]:
def RDD_standardize(vectors_rdd):
    """
    Standardize an RDD of vectors.
    Args:
        vectors_rdd (pyspark.RDD): An RDD where each element is a tuple. The first part of the tuple is an identifier (id, category),
        and the second part is a list or array of numerical features.

    Returns:
        pyspark.RDD: An RDD of standardized vectors where each feature has zero mean and unit variance.
    """
    # flat map values (they are lists), to have a row for each element in the list
    vectors_per_feature_RDD = (vectors_rdd.map(lambda r: (r[0][1], (r[0][0], r[1])))
                    .map(lambda r: (r[0], [((r[1][1][i],r[1][0]),i) for i in range(len(r[1][1]))]))
                    .flatMap(lambda r: [(r[0],el) for el in r[1]])
                    .map(lambda r: ((r[0],r[1][1]), r[1][0])))

    # get a list for each of the features
    values_per_feature_RDD = (vectors_per_feature_RDD.map(lambda r: (r[0], r[1][0]))
                                            .groupByKey().mapValues(list))

    mean_cat_RDD = values_per_feature_RDD.map(lambda r: (r[0], np.mean(r[1])))
    mean_cat_dict = dict(mean_cat_RDD.collect())
    std_cat_RDD = values_per_feature_RDD.map(lambda r: (r[0], np.std(r[1])))
    std_cat_dict = dict(std_cat_RDD.collect())
    mean_cat_dict_br = sc.broadcast(mean_cat_dict)
    std_cat_dict_br = sc.broadcast(std_cat_dict)

    vectors_stand_RDD = (vectors_per_feature_RDD.map(lambda r: (r[0], (r[1][1],(r[1][0]-mean_cat_dict_br.value[r[0]])/std_cat_dict_br.value[r[0]])))
                        .map(lambda r: (r[1][0], (r[0], r[1][1])))
                        .groupByKey().mapValues(list)
                        .mapValues(sorted)
                        .map(lambda r: (r[0], [v[1] for v in r[1]])))
    return vectors_stand_RDD

In [ ]:
vectors_stand_RDD = RDD_standardize(vectors_RDD)

#### Principal-Component Analysis (PCA)

We build the covariance matrix for the data, and we compute its eigenvalues and eigenvectors.

In [ ]:
def outer_prod(v):
    """ Compute the outer product of a vector with itself

    Args:
        v (list or numpy.ndarray): A vector (list or NumPy array) whose outer product with itself is to be computed.

    Returns:
        numpy.ndarray: A 2D NumPy array representing the outer product of the input vector.
    """
    return np.outer(np.array(v), np.array(v))

def PCA(vectors_rdd):
    """
    Perform Principal Component Analysis (PCA) on an RDD of vectors.

    Args:
        vectors_rdd (pyspark.RDD): An RDD where each element is a tuple with an identifier, and the second part is a vector of numerical features.

    Returns:
        tuple: A tuple containing:
            - sorted_eigenvalues (numpy.ndarray): The eigenvalues sorted in non increasing order.
            - sorted_eigenvectors (numpy.ndarray): The eigenvectors corresponding to the sorted eigenvalues.
    """
    n_vectors = vectors_rdd.count()
    cov_matrix = (vectors_rdd.map(lambda r: (r[0], outer_prod(r[1])))
                .reduce(lambda a, b: (1, a[1] + b[1])))[1] / n_vectors

    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

    # sort eigenvalues and eigenvectors in non increasing order
    sorted_indices = np.argsort(eigenvalues)[::-1]
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices]
    return (sorted_eigenvalues, sorted_eigenvectors)

In [ ]:
sorted_eigenvalues, sorted_eigenvectors = PCA(vectors_stand_RDD)

We only retain the components such that their total cumulative explained variance is at least $95\%$.

In [ ]:
def k_principal_components(sorted_eigenvalues, sorted_eigenvectors, t_PCA):
    """
    Select the top-k principal components based on the cumulative explained variance threshold.

    Args:
        sorted_eigenvalues (numpy.ndarray): The eigenvalues sorted in non increasing order.
        sorted_eigenvectors (numpy.ndarray): The eigenvectors corresponding to the sorted eigenvalues.
        t_PCA (float): The cumulative explained variance threshold (a value between 0 and 1).

    Returns:
        numpy.ndarray: The top-k eigenvectors that explain at least the specified threshold of the variance.
    """
    explained_variance = sorted_eigenvalues / np.sum(sorted_eigenvalues)
    cumulative_explained_variance = np.cumsum(explained_variance)
    k = np.argmax(cumulative_explained_variance >= t_PCA) + 1
    top_k_eigenvectors = sorted_eigenvectors[:, :k]
    return top_k_eigenvectors

In [ ]:
t_PCA = 0.95
top_k_eigenvectors = k_principal_components(sorted_eigenvalues, sorted_eigenvectors, t_PCA)
k = top_k_eigenvectors.shape[1]

print(f"Number of components explaining at least {t_PCA*100}% of the variance: {k}")

Number of components explaining at least 95.0% of the variance: 53


In [ ]:
def RDD_dot_product(vectors_rdd, vectors):
    """
    Compute the dot product of each vector in an RDD with a given set of vectors.

    Args:
        vectors_rdd (pyspark.RDD): An RDD where each element is a tuple. The first part is an identifier, and the second part is a vector of numerical features.
        vectors (numpy.ndarray): A 2D NumPy array of vectors with which the dot product is calculated.

    Returns:
    """
    vectors_bc = sc.broadcast(vectors)
    return vectors_rdd.map(lambda r: (r[0], np.dot(vectors_bc.value.T, r[1])))

In [ ]:
vectors_reduced_RDD = RDD_dot_product(vectors_stand_RDD, top_k_eigenvectors)

In [ ]:
vectors_reduced_RDD.first()

(1000040,
 array([25.45800123+0.j,  0.1148132 +0.j,  2.68334325+0.j, -1.46901469+0.j,
         5.13042761+0.j, -3.97129746+0.j, -0.85135626+0.j,  3.21238128+0.j,
         0.56347861+0.j, -3.0955089 +0.j,  1.192826  +0.j, -0.94432451+0.j,
        -0.31643041+0.j, -2.02732624+0.j, -0.60681639+0.j,  1.08076536+0.j,
         3.86072104+0.j,  0.3426764 +0.j,  0.67578278+0.j, -2.36956032+0.j,
        -0.42846868+0.j, -0.46681877+0.j, -1.86640161+0.j, -1.38356922+0.j,
        -1.5635278 +0.j,  2.89316266+0.j,  1.01910017+0.j,  1.29634476+0.j,
         3.00697217+0.j,  0.4618368 +0.j, -0.44399777+0.j,  0.70577557+0.j,
         0.86703545+0.j, -0.61718808+0.j,  0.15544444+0.j, -0.45561834+0.j,
        -0.1218305 +0.j,  2.75527355+0.j,  0.60177256+0.j, -0.04529525+0.j,
        -0.73068582+0.j,  0.31906732+0.j,  2.76736076+0.j,  0.91557656+0.j,
         0.65041097+0.j, -3.13166424+0.j, -1.00339165+0.j, -0.13012935+0.j,
        -1.390677  +0.j, -0.49936246+0.j,  0.37031302+0.j,  0.2827921 +0.j,
  

In [ ]:
def RDD_dimensionality_reduction_PCA(vectors_rdd, t_PCA):
    """
    Perform dimensionality reduction using PCA on an RDD of vectors.

    Args:
        vectors_rdd (pyspark.RDD): An RDD where each element is a tuple. The first part is an identifier, and the second part is a vector of numerical features.
        t_PCA (float): The threshold for cumulative explained variance (a value between 0 and 1).

    Returns:
        pyspark.RDD: An RDD where each element is a tuple containing the identifier and the reduced vector, based on PCA.
    """
    sorted_eigenvalues, sorted_eigenvectors = PCA(vectors_rdd)
    top_k_eigenvectors = k_principal_components(sorted_eigenvalues, sorted_eigenvectors, t_PCA)
    k = top_k_eigenvectors.shape[1]
    vectors_reduced_RDD = RDD_dot_product(vectors_rdd, top_k_eigenvectors)
    return vectors_reduced_RDD, k

An example of reduced vector:

In [ ]:
id, vector = vectors_reduced_RDD.first()
print("movie id:\t {}\n  vector:\t {}".format(id,vector))

movie id:	 1000040
  vector:	 [25.45800123+0.j  0.1148132 +0.j  2.68334325+0.j -1.46901469+0.j
  5.13042761+0.j -3.97129746+0.j -0.85135626+0.j  3.21238128+0.j
  0.56347861+0.j -3.0955089 +0.j  1.192826  +0.j -0.94432451+0.j
 -0.31643041+0.j -2.02732624+0.j -0.60681639+0.j  1.08076536+0.j
  3.86072104+0.j  0.3426764 +0.j  0.67578278+0.j -2.36956032+0.j
 -0.42846868+0.j -0.46681877+0.j -1.86640161+0.j -1.38356922+0.j
 -1.5635278 +0.j  2.89316266+0.j  1.01910017+0.j  1.29634476+0.j
  3.00697217+0.j  0.4618368 +0.j -0.44399777+0.j  0.70577557+0.j
  0.86703545+0.j -0.61718808+0.j  0.15544444+0.j -0.45561834+0.j
 -0.1218305 +0.j  2.75527355+0.j  0.60177256+0.j -0.04529525+0.j
 -0.73068582+0.j  0.31906732+0.j  2.76736076+0.j  0.91557656+0.j
  0.65041097+0.j -3.13166424+0.j -1.00339165+0.j -0.13012935+0.j
 -1.390677  +0.j -0.49936246+0.j  0.37031302+0.j  0.2827921 +0.j
 -2.77411858+0.j]


## Locality Sensitive Hashing (LSH)

### Locality sentitive family for cosine distance

We build the locality sensitive family $\mathbf{F}$ as a set of randomly chosen vectors $\{v_{f\in\mathbf{F}}\}$. Given two vectors $x$ and $y$, they make a candidate pair of similar items if and only if the dot products $x\cdot v_f$ and $x \cdot v_f$ have the same sign. A family of functions $\mathbf{F}$ built as described is a locality-sensitive family for the cosine distance.

We will also refer to the random vectors in $\mathbf{F}$ as hash functions.

Since we will compute the dot product between each of the elements in the dataset and all of the hash functions in $\mathbf{F}$, we try to simplify the computation of the cosine distance between vectors. We do so by restricting the random choice of vectors to those having components $+1$ or $-1$. Hence the dot product of any vector $x$ with a vector in such a family $\mathbf{F}$ is given by its algebraic sum $x$'s components, where the signs depend on the components of the random vector.

In [ ]:
def RDD_LS_hash_family_cosine_distance(signature_length, num_components):
    """
    Generate a locality sentitive family for cosine distance.

    Args:
        signature_length (int): The signature length for the items.
        num_components (int): he number of components for the random vectors.
    Returns:
        pyspark.RDD: An RDD of hash functions, where each hash function is a tuple with an index and a random vector in {-1,1}^k.
    """

    hash_funcs_RDD = (sc.parallelize([(i,1) for i in range(signature_length)])
                    .map(lambda r: (r[0],np.random.choice([-1, 1], num_components))))

    return hash_funcs_RDD

For each of the vectors in $\mathbf{F}$ (called `hash_funcs_RDD` in the code), we compute the dot products with each of the (reduced) vectors in the dataset.

In [ ]:
def RDD_signatures(vectors_rdd, hash_funcs_rdd):
    """
    Generate locality-sensitive hashing (LSH) signatures for a set of vectors.

    Args:
        vectors_rdd (pyspark.RDD): An RDD where each element is a tuple. The first part is an identifier, and the second part is a vector of numerical features.
        signature_length (int): The signature length for the items.
    Returns:
        pyspark.RDD: An RDD where each element is a tuple containing an identifier and its corresponding LSH signature.
    """
    vectors_hash_RDD = (vectors_rdd.cartesian(hash_funcs_rdd)
                    .map(lambda r: ((r[0][0],r[1][0]), (r[0][1], r[1][1]))))
    signatures_RDD = (vectors_hash_RDD.map(lambda r: (r[0], np.sign(np.dot(r[1][0], r[1][1]))))
                    .map(lambda r: (r[0][0], (r[0][1], r[1])))
                    .groupByKey().mapValues(list)
                    .mapValues(sorted)
                    .map(lambda r: (r[0], [int(v[1]) for v in r[1]])))
    return signatures_RDD

In [ ]:
signature_length=100
hash_funcs_RDD = RDD_LS_hash_family_cosine_distance(signature_length, k)
signatures_RDD = RDD_signatures(vectors_reduced_RDD, hash_funcs_RDD)

Let's give look at a possibile signature for a movie.

In [ ]:
id, signature = signatures_RDD.first()
print(" movie id:\t {}\nsignature:\t {}".format(id,signature))

 movie id:	 1000086
signature:	 [-1, -1, -1, 1, -1, -1, -1, -1, -1, 1, 1, -1, 1, -1, -1, -1, -1, 1, -1, 1, 1, -1, -1, 1, -1, 1, 1, 1, 1, -1, -1, 1, 1, -1, -1, 1, 1, -1, -1, 1, 1, 1, 1, 1, 1, -1, 1, 1, -1, 1, -1, -1, 1, 1, -1, -1, -1, 1, 1, -1, -1, -1, -1, 1, 1, -1, 1, -1, 1, 1, 1, -1, 1, 1, -1, -1, -1, -1, -1, 1, 1, 1, -1, -1, 1, -1, -1, 1, 1, -1, -1, -1, 1, -1, -1, -1, -1, -1, -1, 1]


Now, `signatures_RDD` contains the so called *signature matrix*, that contains a signature for each movie in the dataset.

### Banding technique

It would be unthinkable to compare all possible pairs of movies to find similar ones among them. This would mean scanning all the rows in the signature matrix to compute the relative frequency between possible pairs of movies. So, we proceed by applying locality-sensitive hashing. In this approach we reduce the number of rows that determine the signature of a movie by hashing so called bands of rows. The rationale is that similar movies are more likely to be hashed in the same bucket, so we hope that dissimilar pairs end up in distinct buckets, and thus are never checked for similarity. Looking at the resulting signatures we consider as a candidate pair only those for which their cosine similarity exceeds a threshold $t$.

We begin by dividing the signature matrix into $b$ bands of $r$ rows each. The choice of $r$ and $b$ depends on the threshold $t$ on the cosine distance between pairs of movies. The value of the threshold $t$ is approximately the value of similarity at which the probability of becoming a candidate is $\frac{1}{2}$.
So, we keep into account the following relationships (for brevity $l$=`signature_length`):   

\begin{equation*}
    \begin{cases}
        t=\left(\frac{1}{b}\right)^\frac{1}{r} \\
        r\cdot b = l
    \end{cases}
\end{equation*}

The function `band_size` solves this system in $r$ by computing it's value through
\begin{equation*}
    r=-\frac{W(-l\ln(t))}{\ln(t)}
\end{equation*}

where $W$ is the Lambert $W$ function, used to solve equations in the form $we^{w}=z$ for $w$.

In [ ]:
from scipy.special import lambertw
import math

def band_size(t, signature_length):
    """
    Compute the number of rows that form a band for the LSH technique.

    Args:
        t (int): The desired threshold in [0,1] on the similarity between pairs of items.
        signature_length (int): The signature length for the items.

    Returns:
        int: The ideal number of rows contained in the band.
    """
    return -math.ceil(lambertw(-signature_length*np.log(t)).real/np.log(t))

def r_b_choice(t,signature_length):
    """
    Choose the adeguate number of rows a band and the number of bands for the LSH technique.

    Args:
        t (int): The desired threshold in [0,1] on the similarity between pairs of items.
        signature_length (int): The signature length for the items.

    Returns:
        int: The ideal number of rows, and consequent number of bands on the basis of the signature length.
    """
    r=band_size(t,signature_length)
    b=math.ceil(signature_length/r)
    return (r,b)

In [ ]:
t=0.8
t_bc = sc.broadcast(t)
r,b = r_b_choice(t,signature_length)
print("The chosen parameters are: \n r: {} \n b: {}".format(r,b))

The chosen parameters are: 
 r: 10 
 b: 10


Having chosen the parameters we proceed by subdividing the rows of the similarity matrix into bands.

In [ ]:
def split_list(l,b):
    """ Split list into b lists of equal length

    Args:
        l (list): list of elements
        b (int): number of sublists

    Returns:
        list: list formed by b sublists all of the same length, except for the last one if len(l) is not a multiple of b
    """
    split_list = [(list(a)) for a in np.array_split(np.array(l), b)]
    return [split_list[i] for i in range(b)]

We proceed by hashing the rows in each of the $b$ bands for each vector. For each band we use a different bucket array, so that signatures with two equal vectors in separate bands are hashed differently. This is done by setting a hash function with a distinct seed for all bands.

In [ ]:
def RDD_LSH(signatures_rdd, b, hash_bucket_size):
    """ Compute the hashed signatures with the LSH technique

    Args:
        signatures_rdd (RDD): RDD of (set key, signature for set)
        b (int): number of bands in which to split the signature matrix represented by signatures_rdd
        hash_bucket_size (int): the size of the bucket to which the signature portions in each band get hashed to

    Returns:

    """
    b_bc = sc.broadcast(b)
    hash_bucket_size_bc = sc.broadcast(hash_bucket_size)
    return (signatures_rdd
            .map(lambda r: (r[0],split_list(r[1],b_bc.value)))
            .map(lambda r: (r[0],[hash_object(bytes(str(t),'ascii'),i,hash_bucket_size_bc.value) for i,t in enumerate(r[1])])))

Also, to avoid hashing distinct portions of a signature in the same bucket it is important to choose a great enough bucket. Here we have evaluted the number of tuples given by $\{-1,1\}^r$.

In [ ]:
hash_bucket_size = 2**20
hashed_signatures_RDD = RDD_LSH(signatures_RDD, b, hash_bucket_size)

Let's give a look at the new compact representation of a movie.

In [ ]:
id, signature = hashed_signatures_RDD.first()
print(" movie id:\t {}\nsignature:\t {}".format(id,signature))

 movie id:	 1000040
signature:	 [223957, 4666, 69241, 1046797, 397796, 744068, 180206, 477230, 131910, 596412]


### Find similar items

Now we search for candidate pairs among the reviews. We consider as possible similar couples of reviews those that have cosine similarity at least $t$.

In [ ]:
def RDD_pairs(rdd):
    """ Put together all possible pairs of rows, without repetitions

    Args:
        rdd (RDD): RDD of (row key, information regarding row)

    Returns:
        RDD: RDD of ((r1,r2),(info1, info2)), with r1>r2 to avoid having duplicates
    """
    pairs_RDD = rdd.cartesian(rdd).filter(lambda r: r[0][0] > r[1][0])
    return pairs_RDD.map(lambda r: ((r[0][0], r[1][0]),(r[0][1], r[1][1])))



def RDD_candidate_pairs(rdd):
    """ Filter from pairs of rows whether they are candidate pairs

    Args:
        rdd (RDD): RDD of (row key, information regarding the row)

    Returns:
        RDD: RDD of ((r1,r2),(info1, info2)) such that exists i such that el_r1[i]==el_r2[i]
    """
    pairs_RDD = RDD_pairs(rdd)
    return pairs_RDD.filter(lambda r: any(x == y for x, y in zip(r[1][0], r[1][1])))

def cosine_distance(vec1,vec2):
    """
    Compute the cosine distance between two vectors.

    Args:
        vec1 (numpy.ndarray): The first vector.
        vec2 (numpy.ndarray): The second vector.

    Returns:
        float: The cosine distance between the two vectors, in radians.
    """
    cosine = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    return np.arccos(cosine)

def cosine_similarity(vec1,vec2):
    """
    Compute the cosine similarity between two vectors.

    Args:
        vec1 (numpy.ndarray): The first vector.
        vec2 (numpy.ndarray): The second vector.

    Returns:
        float: The cosine similarity between the two vectors.
    """

    return (math.pi-cosine_distance(vec1,vec2))/math.pi

def RDD_similar_items(rdd,t, similarity_function):
    """ Filter from pairs of rows whether they have similarity at least t

    Args:
        rdd (RDD): RDD of (row key, information regarding the row)
        t (int): desired threshold on the Jaccard similarity
        similarity_function (function): function to compute the similarity between two vectors

    Returns:
        RDD: RDD of ((k1,k2),1) if similarity_function(info1,info2)>=t
    """
    t_bc = sc.broadcast(t)
    return (rdd
            .filter(lambda r: similarity_function(r[1][0],r[1][1])>=t_bc.value)
            .map(lambda r: (r[0],similarity_function(r[1][0],r[1][1]))))

In [ ]:
candidate_pairs_RDD = RDD_candidate_pairs(hashed_signatures_RDD).cache()
similar_items_RDD = RDD_similar_items(candidate_pairs_RDD,t, cosine_similarity).cache()

## Experiments

In this paragraph we run some experiments with various combinations of the parameters. We have followed the outline described in Section 3.2.2 of the report.

In [ ]:
def data_pre_processing(data_rdd, t_PCA):
    ## we do no repeat steps that do not depend on the parameters for computational purposes
    # * polish data
    # * data embedding
    # * manage attributes with multiple values
    print("Standardizing vectors...")
    vectors_stand_RDD = RDD_standardize(data_rdd)
    print("Dimensionality reduction...")
    vectors_reduced_RDD, k = RDD_dimensionality_reduction_PCA(vectors_stand_RDD, t_PCA)
    return vectors_reduced_RDD, k

def find_similar_items(data_rdd, k, l, t, b, h):
    print("Creating locality sensitive family...")
    hash_funcs_RDD = RDD_LS_hash_family_cosine_distance(l, k).cache()
    print("Building signatures...")
    signatures_RDD = RDD_signatures(data_rdd, hash_funcs_RDD).cache()
    print("Applying banding technique...")
    hashed_signatures_RDD = RDD_LSH(signatures_RDD, b, h).cache()
    print("Finding candidate pairs...")
    candidate_pairs_RDD = RDD_candidate_pairs(hashed_signatures_RDD).cache()
    print("Filtering similar items...")
    similar_items_RDD = RDD_similar_items(candidate_pairs_RDD, t, cosine_similarity)
    return similar_items_RDD, candidate_pairs_RDD

def run_experiment(data_rdd, t_PCA, l, t, b, h):
    vectors_RDD, k = data_pre_processing(data_rdd, t_PCA)
    print(f"Number of components explaining at least {t_PCA*100}% of the variance: {k}")
    similar_items_RDD, candidate_pairs_RDD = find_similar_items(vectors_RDD, k, l, t, b, h)
    return similar_items_RDD, candidate_pairs_RDD

def save_results(similar_items_rdd, candidate_pairs_rdd, exp_name, *parameters):
    data = {
        "parameters": parameters,
        "similar_items": similar_items_rdd.collect(),
        "candidate_pairs": candidate_pairs_rdd.collect()
    }
    with open(f"results_{exp_name}.json", "w") as f:
        json.dump(data, f)

### Experiment 1

In [ ]:
parameters = (
    0.95,               #t_PCA
    100,                #l
    0.5,                #t
    2**20,              #h
)
parameters += (r_b_choice(parameters[2],parameters[1])[1],) #b

similar_items_RDD, candidate_pairs_RDD = run_experiment(vectors_RDD, *parameters)
print("Saving results...")
similar_items_RDD = similar_items_RDD.coalesce(1)
candidate_pairs_RDD = candidate_pairs_RDD.coalesce(1)
candidate_pairs_RDD.saveAsTextFile(f"candidate_pairs_exp_2.txt")
similar_items_RDD.saveAsTextFile(f"similar_items_exp_2.txt")
print(parameters)
with open(f"parameters_exp_2.json", "w") as f:
    json.dump(parameters, f)

Standardizing vectors...


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "spark-3.5.3-bin-hadoop3/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "spark-3.5.3-bin-hadoop3/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 